In [1]:
%cd ..
%load_ext autoreload
%autoreload 2

/Users/martinbierey/git/dax-hoc


In [2]:
from db.models import Company, EarningsEvent, ExpectationsSnapshot
from src.expectations.researcher import research,_PROMPT,call_llm,_parse_sections
from src.db import read_all,get_engine
from sqlalchemy.dialects.sqlite import insert
import uuid
from datetime import date, datetime, timezone
import json

/Users/martinbierey/git/dax-hoc/.venv/lib/python3.14/site-packages/anthropic/_compat.py:48: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.typing import (


In [3]:
engine = get_engine()

In [4]:
dfs = read_all()

In [5]:
event = EarningsEvent(
        fiscal_period="2026-Q1",
        event_type="quarterly",
        expected_date=date(2026, 5,13),
        time_confidence="exact",
        source="yahoo_finance",
        status="scheduled",
        actual_release_at=None,
        news_item_id=None,
        last_synced_at=datetime.now(timezone.utc),
    )

company = Company(
        isin="DE000A2LQ884",
        name="AUTO1 GROUP SE",
        index_name="DAX",
        industry='Automotive E-Commerce',
        description="Europe's leading digital automotive marketplace for used cars operating Autohero (B2C retail) and AUTO1.com (B2B trade) platforms.",
        synced_at=datetime.now(timezone.utc),
    )

In [6]:
prompt = _PROMPT.format(
        company_name=company.name,
        isin=company.isin,
        index_name=company.index_name,
        industry=company.industry or "N/A",
        fiscal_period=event.fiscal_period,
        event_type=event.event_type,
        expected_date=str(event.expected_date),
    )

content, run_id = call_llm(
        role="expectations_researcher",
        prompt=prompt,
        engine=engine,
        earnings_event_id=event.id,
        web_search=True,
    )

narrative_md, trade_thesis_md, sources = _parse_sections(content)
snapshot = ExpectationsSnapshot(
        id=str(uuid.uuid4()),
        earnings_event_id=event.id,
        gathered_at=datetime.now(timezone.utc),
        narrative_md=narrative_md,
        trade_thesis_md=trade_thesis_md,
        sources_json=json.dumps(sources),
        raw_research_text=content,
        llm_run_id=run_id,
    )

In [8]:
snapshot.narrative_md

'The market narrative for AUTO1 in 2026 is defined by a "show-me" story regarding unit economics. While the company delivered record 2025 results, the stock faced selling pressure after management issued 2026 guidance that implied a compression in **Gross Profit per Unit (GPU)** compared to the exit rates of late 2025. Investors are currently skeptical of whether the company can maintain its high profitability margins while pursuing its aggressive goal of 940,000 to 1,000,000 units for the full year.\n\n*   **Consensus Expectations:** Analysts are looking for Q1 revenue of approximately **€2.28B – €2.29B** and an EPS of **€0.16 – €0.18**.\n*   **The GPU Tension:** In Q4 2025, AUTO1 achieved a group GPU of €1,211. However, the FY 2026 guidance (midpoint of €1.15B Gross Profit / 970k units) implies a drop to ~€1,185. Q1 must prove this was merely "conservative guidance" rather than a fundamental shift in used-car pricing power.\n*   **Financing as a Catalyst:** The market is closely watc

In [9]:
snapshot.trade_thesis_md

'Buying the stock post-announcement is recommended if the company demonstrates that it is not sacrificing margin for volume, effectively "debunking" the conservative guidance that weighed on the stock earlier in the year.\n\n**Concrete Buy Conditions:**\n1.  **Group GPU Beat:** Total Group Gross Profit per Unit must come in at **>€1,215**. This would exceed the Q4 2025 exit rate and prove that the 2026 guidance was indeed overly conservative, signaling that margins are expanding or stable, not contracting.\n2.  **Autohero Momentum:** Retail (Autohero) units must grow **>35% YoY** (reaching ~30,000 units for the quarter) with a **Retail GPU >€2,650**. Because Retail is the higher-margin engine, a beat here provides the necessary operating leverage to hit the upper end of the €250M–€275M Adj. EBITDA guidance.\n3.  **Positive Revision Language:** Management must explicitly confirm that they are trending toward the **upper half** of the FY 2026 Adjusted EBITDA range (€262M+).\n\n**Deal-Bre